# Read audit, physics and trace

## Goal
PASS means the configured check passed. WARN, FAIL and UNKNOWN retain their meaning. File/physics acceptance does not establish operating-policy quality. Requested release and implemented total outflow are both recorded; next-day history contains the physical outcome.

## Setup
Python 3.10+ with repository dependencies. All cells use local files; no provider calls.

In [1]:
from pathlib import Path
import json, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/hydrocnhs').is_dir())
sys.path.insert(0, str(ROOT/'src'))
from llm_reservoir_agent.workflow import load_settings, run_settings, verify_settings_output
from llm_reservoir_agent.verification import verify_run
from unittest.mock import patch
def ensure_offline(config):
    settings = load_settings(ROOT/config)
    assert settings.report['planned_llm_calls'] == 0
    output = Path(settings.report['output'])
    if not output.exists():
        with patch('urllib.request.urlopen', side_effect=AssertionError('network forbidden')):
            run_settings(settings)
    return settings, verify_settings_output(settings)
print('Checkout:', ROOT.name)


Checkout: clean-upgrade-99ea43e42a


## Steps

### Read the decision-to-physics chain

In [2]:
settings, verified=ensure_offline('examples/fe_walter/dkp/experiment.tutorial.yaml')
output=Path(settings.report['output'])
physics=[json.loads(line) for line in (output/'physics.jsonl').read_text().splitlines()]
first=physics[0]
print('Binding:', {k:first[k] for k in ['decision_id','reservoir_id','timestep']})
print('Physics:', first['record']['transition'])
audit=[json.loads(line) for line in (output/'audit.jsonl').read_text().splitlines()]
print('Audit statuses:', [row['status'] for row in audit])
print('Policy accepted:', verified['operating_strategy_accepted'])

Binding: {'decision_id': 'f833358bd1c71a188e4d5c51b3709f31d9be01750ec4ed51776be5754666ea07', 'reservoir_id': 'fe_walter', 'timestep': 0}
Physics: {'adjustments': [], 'controlled_release_cms': 32.12531870666853, 'inflow_cms': 32.12531870666853, 'mass_balance_residual_m3': 0.0, 'minimum_release_shortfall_cms': 0.0, 'spill_cms': 0.0, 'storage_end_m3': 2457848.279729185, 'storage_start_m3': 2457848.279729185, 'target_release_cms': 32.12531870666853, 'total_outflow_cms': 32.12531870666853}
Audit statuses: ['PASS', 'PASS', 'PASS']
Policy accepted: None


### Inspect actual next-day history

In [3]:
raw=[json.loads(p.read_text()) for p in (output/'raw').glob('policy_*.json')]
variables=sorted((p['backend_trace']['input_variables'] for p in raw),key=lambda p:p['timestep'])
print('Next-day history:', variables[1]['native_history'])
print('Residual m3:',verified['max_system_balance_residual_m3'])

Next-day history: [{'date': '2016-01-01', 'decision_release_cfs': 1134.4949071013637, 'implemented_release_cfs': 1134.4949071013637, 'inflow_cfs': 1134.4949071013637, 'storage_end_af': 1992.6100893658788}]
Residual m3: 0.0


## Checks
Review the printed inputs, counts, source labels and residuals. Synthetic replies validate interfaces only.

## Next steps
Malformed/incomplete replies stop before physical release and are not retried automatically. See docs/auditor_plugins.md to add a versioned reservoir-specific checker.